# Step 1 — simulate

A later notebook: it does **not** take the varying values directly, it loads them
from `settings.json` (whose path nb2slurm passes in). It runs the Monte Carlo
estimate and saves the result under this job's `outdir`.

In [ ]:
settings_path = "settings.json"

In [ ]:
import json
import sys
from pathlib import Path
import nb2slurm

# The helper lives in scripts/. On the cluster the job runs from the project root,
# so it imports directly; run locally this notebook lives in notebooks/, so add the
# project root to the path. on_hpc() tells the two apart.
if not nb2slurm.on_hpc():
    sys.path.append(str(Path().resolve().parent))
from scripts.montecarlo import estimate_pi

settings = nb2slurm.Settings.load(settings_path)
outdir = Path(settings["outdir"])

In [ ]:
result_file = outdir / "result.json"

# idempotent: don't recompute if this job already finished
if result_file.exists():
    result = json.loads(result_file.read_text())
    print(f"reusing existing result: pi ~= {result['pi']}")
else:
    pi, inside, points = estimate_pi(settings["n_samples"], settings["seed"])
    result = {"pi": pi, "inside": inside, "n_samples": settings["n_samples"],
              "seed": settings["seed"]}
    result_file.write_text(json.dumps(result, indent=2))
    # save the points so the plot step can draw them
    (outdir / "points.json").write_text(json.dumps(points))
    print(f"n_samples={settings['n_samples']} seed={settings['seed']} -> pi ~= {pi}")